# Iceberg V3 Basics - VARIANT & Nanosecond Timestamps

## Test Case: v3 Data Types (Private Preview)
- **Capability**: Semi-structured data in v3 format  
- **Target**: Query performance within 15% of Native tables
- **Features**: VARIANT, TIMESTAMP_NTZ(9), nested JSON

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE SCHEMA TESTS;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- Test 1: Create Iceberg v3 table with VARIANT and nanosecond timestamps
CREATE OR REPLACE ICEBERG TABLE ICEBERG_POC.TESTS.V3_VARIANT_TEST (
    id INT,
    created_at TIMESTAMP_NTZ(9),
    payload VARIANT,
    metadata VARIANT
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'EXVOL'
ICEBERG_VERSION = 3;

In [ ]:
-- Test 2: Insert 1M rows with nested JSON
INSERT INTO ICEBERG_POC.TESTS.V3_VARIANT_TEST
SELECT 
    SEQ4() AS id,
    DATEADD('nanosecond', UNIFORM(0, 999999999, RANDOM()), CURRENT_TIMESTAMP())::TIMESTAMP_NTZ(9) AS created_at,
    OBJECT_CONSTRUCT(
        'user', OBJECT_CONSTRUCT(
            'name', 'User_' || SEQ4(),
            'email', 'user' || SEQ4() || '@example.com',
            'preferences', OBJECT_CONSTRUCT(
                'theme', ARRAY_CONSTRUCT('dark', 'light')[UNIFORM(0, 1, RANDOM())],
                'notifications', UNIFORM(0, 1, RANDOM())::BOOLEAN
            )
        ),
        'items', ARRAY_CONSTRUCT(
            OBJECT_CONSTRUCT('sku', 'SKU-' || UNIFORM(1000, 9999, RANDOM()), 'qty', UNIFORM(1, 10, RANDOM())),
            OBJECT_CONSTRUCT('sku', 'SKU-' || UNIFORM(1000, 9999, RANDOM()), 'qty', UNIFORM(1, 10, RANDOM()))
        )
    ) AS payload,
    OBJECT_CONSTRUCT('source', 'poc_test', 'version', '3.0') AS metadata
FROM TABLE(GENERATOR(ROWCOUNT => 1000000));

In [ ]:
-- Test 3: Query nested VARIANT data - measure performance
SELECT 
    id,
    TO_VARCHAR(created_at, 'YYYY-MM-DD HH24:MI:SS.FF9') AS timestamp_ns,
    payload:user:name::STRING AS user_name,
    payload:user:preferences:theme::STRING AS theme,
    payload:items[0]:sku::STRING AS first_item_sku,
    ARRAY_SIZE(payload:items) AS item_count
FROM ICEBERG_POC.TESTS.V3_VARIANT_TEST
WHERE payload:user:preferences:notifications = TRUE
LIMIT 100;

In [ ]:
-- Test 4: Aggregations on VARIANT data
SELECT 
    payload:user:preferences:theme::STRING AS theme,
    COUNT(*) AS user_count,
    AVG(ARRAY_SIZE(payload:items)) AS avg_items
FROM ICEBERG_POC.TESTS.V3_VARIANT_TEST
GROUP BY 1;

In [ ]:
-- Verify table metadata
SHOW ICEBERG TABLES LIKE 'V3_VARIANT_TEST' IN SCHEMA ICEBERG_POC.TESTS;

SELECT COUNT(*) AS total_rows FROM ICEBERG_POC.TESTS.V3_VARIANT_TEST;